# 3. Boiling point: frozen MoLFormer embeddings + feed-forward network

**Workshop goal:** use a pretrained molecular language model as a fixed feature extractor, then train a small feed-forward network to predict boiling point. MoLFormer reads each SMILES string and produces an embedding. In this notebook the large pretrained transformer stays frozen; only the regression head learns from the boiling-point labels.

[![Overview of the MoLFormer pretraining and downstream-transfer workflow](https://media.github.ibm.com/user/4935/files/594363e6-497b-4b91-9493-36ed46f623a2)](https://github.com/IBM/molformer)

*Figure 1. MoLFormer pretraining and downstream transfer workflow, from the [IBM MoLFormer repository](https://github.com/IBM/molformer). The workshop uses the pretrained SMILES encoder as a frozen embedding generator and trains only the small feed-forward boiling-point head.*

## Workshop workflow

1. Recreate the same reproducible training, validation, and held-out test split used in notebook 02.
2. Convert the split SMILES strings into frozen MoLFormer embeddings.
3. Train a feed-forward regression network on the training embeddings and select the epoch with the lowest validation loss.
4. Evaluate that selected model once on the held-out test set with a loss plot, parity plot, and good/bad molecular examples.
5. Use the final helper function to predict boiling point for a new SMILES string.

The identical random split makes the Chemprop and MoLFormer demonstrations comparable. It is suitable for this CPU-sized workshop exercise; a scaffold split and repeated cross-validation are sensible next steps for a scientific performance claim.

## Key MoLFormer references

1. [Ross et al. (2022), *Large-scale chemical language representations capture molecular structure and properties*](https://doi.org/10.1038/s42256-022-00580-7) — the MoLFormer architecture, pretraining strategy, and benchmark study.
2. [Ross et al. (2021), *Large-Scale Chemical Language Representations Capture Molecular Structure and Properties*](https://arxiv.org/abs/2106.09553) — open preprint and technical details.
3. [IBM MoLFormer repository](https://github.com/IBM/molformer) — reference implementation, pretrained resources, and embedding-extraction examples.

In [ ]:
import copy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from rdkit import Chem
from rdkit.Chem import Draw
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

# Use the same data-selection and split settings as notebook 02.
SEED = 42
SAMPLE_SIZE = 4000
MODEL_ID = 'ibm-research/MoLFormer-XL-both-10pct'
EMBEDDING_BATCH_SIZE = 32
HEAD_BATCH_SIZE = 64
EPOCHS = 100
PATIENCE = 20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
HIDDEN_DIMENSIONS = (512, 128)
DROPOUT = 0.10
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ACQUIRED_DATA_FILE = Path('outputs/nist_acquisition/NIST_BP_SMILES.parquet')
PACKAGED_DATA_FILE = Path('data/CompoundData.parquet')
DATA_FILE = ACQUIRED_DATA_FILE if ACQUIRED_DATA_FILE.exists() else PACKAGED_DATA_FILE
OUTPUT = Path('outputs/bp_molformer')
FIGURES = OUTPUT / 'figures'
OUTPUT.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

raw_data = pd.read_parquet(DATA_FILE).copy()
required_columns = {'SMILES', 'BP'}
missing_columns = required_columns - set(raw_data.columns)
if missing_columns:
    raise ValueError(f'Missing required columns: {sorted(missing_columns)}')

data = raw_data.dropna(subset=['SMILES', 'BP']).drop_duplicates('SMILES')[['SMILES', 'BP']]
data = data.sample(n=min(SAMPLE_SIZE, len(data)), random_state=SEED).reset_index(drop=True)
invalid_smiles = data.SMILES.map(Chem.MolFromSmiles).isna().sum()
if invalid_smiles:
    raise ValueError(f'Sampled data contain {invalid_smiles} invalid SMILES strings.')

# This is intentionally identical to the split recipe in notebook 02: 80% / 10% / 10%.
indices = np.arange(len(data))
train_indices, held_out_indices = train_test_split(indices, test_size=0.20, random_state=SEED)
val_indices, test_indices = train_test_split(held_out_indices, test_size=0.50, random_state=SEED)
data['split'] = 'train'
data.loc[val_indices, 'split'] = 'val'
data.loc[test_indices, 'split'] = 'test'
if set(data['split']) != {'train', 'val', 'test'} or data.SMILES.duplicated().any():
    raise ValueError('Split validation failed: each unique molecule must have exactly one split.')

train_data = data.loc[data.split == 'train'].reset_index(drop=True)
val_data = data.loc[data.split == 'val'].reset_index(drop=True)
test_data = data.loc[data.split == 'test'].reset_index(drop=True)
data.to_csv(OUTPUT / 'bp_data.csv', index=False)
print(f'Using data source: {DATA_FILE} on {DEVICE}.')
display(data.groupby('split')['BP'].agg(n='size', minimum_K='min', mean_K='mean', maximum_K='max').reindex(['train', 'val', 'test']).round(2))

In [ ]:
try:
    from transformers import AutoModel, AutoTokenizer
except ImportError as error:
    raise ImportError('Install the workshop environment; it includes transformers.') from error

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
encoder = AutoModel.from_pretrained(MODEL_ID, deterministic_eval=True, trust_remote_code=True).to(DEVICE).eval()
for parameter in encoder.parameters():
    parameter.requires_grad = False

@torch.inference_mode()
def embed_smiles(smiles_list, description='Creating frozen embeddings'):
    """Return one fixed MoLFormer embedding per SMILES string."""
    vectors = []
    for start in tqdm(range(0, len(smiles_list), EMBEDDING_BATCH_SIZE), desc=description):
        batch = smiles_list[start:start + EMBEDDING_BATCH_SIZE]
        tokens = tokenizer(batch, padding=True, truncation=True, max_length=202, return_tensors='pt').to(DEVICE)
        outputs = encoder(**tokens)
        pooled = getattr(outputs, 'pooler_output', None)
        if pooled is None:
            hidden = outputs.last_hidden_state
            mask = tokens['attention_mask'].unsqueeze(-1).to(hidden.dtype)
            pooled = (hidden * mask).sum(1) / mask.sum(1).clamp_min(1)
        vectors.append(pooled.float().cpu().numpy())
    return np.vstack(vectors)

train_embedding = embed_smiles(train_data.SMILES.tolist(), 'Embedding training molecules')
val_embedding = embed_smiles(val_data.SMILES.tolist(), 'Embedding validation molecules')
test_embedding = embed_smiles(test_data.SMILES.tolist(), 'Embedding test molecules')

# Fit all preprocessing quantities using training data only.
embedding_mean = train_embedding.mean(axis=0, keepdims=True)
embedding_std = train_embedding.std(axis=0, keepdims=True).clip(1e-6)
target_mean = float(train_data.BP.mean())
target_std = float(train_data.BP.std())

def prepare_embeddings(values):
    return ((values - embedding_mean) / embedding_std).astype(np.float32)

def prepare_targets(values):
    return ((np.asarray(values, dtype=np.float32) - target_mean) / target_std).reshape(-1, 1)

train_dataset = TensorDataset(torch.from_numpy(prepare_embeddings(train_embedding)), torch.from_numpy(prepare_targets(train_data.BP)))
val_dataset = TensorDataset(torch.from_numpy(prepare_embeddings(val_embedding)), torch.from_numpy(prepare_targets(val_data.BP)))
test_dataset = TensorDataset(torch.from_numpy(prepare_embeddings(test_embedding)), torch.from_numpy(prepare_targets(test_data.BP)))
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=HEAD_BATCH_SIZE, shuffle=True, generator=loader_generator)
val_loader = DataLoader(val_dataset, batch_size=HEAD_BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=HEAD_BATCH_SIZE, shuffle=False)
print('Frozen embedding shapes:', train_embedding.shape, val_embedding.shape, test_embedding.shape)

In [ ]:
class FrozenEmbeddingRegressor(nn.Module):
    """A feed-forward regression head; the MoLFormer encoder remains frozen."""
    def __init__(self, input_dim, hidden_dimensions=HIDDEN_DIMENSIONS, dropout=DROPOUT):
        super().__init__()
        layers = [nn.LayerNorm(input_dim)]
        previous_dim = input_dim
        for hidden_dim in hidden_dimensions:
            layers.extend([nn.Linear(previous_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout)])
            previous_dim = hidden_dim
        layers.append(nn.Linear(previous_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, embeddings):
        return self.network(embeddings)

head = FrozenEmbeddingRegressor(train_embedding.shape[1]).to(DEVICE)
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(head.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

@torch.inference_mode()
def mean_loss(loader):
    head.eval()
    total_loss, total_items = 0.0, 0
    for embeddings, targets in loader:
        embeddings, targets = embeddings.to(DEVICE), targets.to(DEVICE)
        loss = criterion(head(embeddings), targets)
        total_loss += loss.item() * len(targets)
        total_items += len(targets)
    return total_loss / total_items

history = []
best_validation_loss = float('inf')
best_epoch = 0
best_state = None
epochs_without_improvement = 0

for epoch in tqdm(range(1, EPOCHS + 1), desc='Training feed-forward head'):
    head.train()
    total_train_loss, total_train_items = 0.0, 0
    for embeddings, targets in train_loader:
        embeddings, targets = embeddings.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(head(embeddings), targets)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item() * len(targets)
        total_train_items += len(targets)

    train_loss = total_train_loss / total_train_items
    validation_loss = mean_loss(val_loader)
    scheduler.step(validation_loss)
    history.append({'epoch': epoch, 'train_loss': train_loss, 'validation_loss': validation_loss,
                    'learning_rate': optimizer.param_groups[0]['lr']})

    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        best_epoch = epoch
        best_state = copy.deepcopy(head.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f'Early stopping after epoch {epoch}; no validation improvement for {PATIENCE} epochs.')
            break

if best_state is None:
    raise RuntimeError('Training did not produce a validation checkpoint.')
head.load_state_dict(best_state)
history = pd.DataFrame(history)
history.to_csv(OUTPUT / 'training_history.csv', index=False)
torch.save({'head_state_dict': head.state_dict(), 'input_dimension': train_embedding.shape[1],
            'hidden_dimensions': HIDDEN_DIMENSIONS, 'embedding_mean': embedding_mean,
            'embedding_std': embedding_std, 'target_mean': target_mean, 'target_std': target_std,
            'model_id': MODEL_ID, 'best_epoch': best_epoch, 'best_validation_loss': best_validation_loss},
           OUTPUT / 'best_frozen_molformer_head.pt')
print(f'Restored the epoch-{best_epoch} head with the lowest validation loss: {best_validation_loss:.4f}.')

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(history.epoch, history.train_loss, marker='o', color='#4472C4', label='Training loss')
ax.plot(history.epoch, history.validation_loss, marker='s', color='#C00000', label='Validation loss')
validation_increases = history.loc[history.validation_loss.diff() > 0]
if not validation_increases.empty:
    ax.scatter(validation_increases.epoch, validation_increases.validation_loss, marker='x', s=80, linewidths=2,
               color='#ED7D31', zorder=4, label='Validation loss increased')
best_row = history.loc[history.validation_loss.idxmin()]
ax.axvline(best_row.epoch, color='#70AD47', linestyle=':', linewidth=2, label='Lowest validation loss')
ax.scatter(best_row.epoch, best_row.validation_loss, s=80, color='#70AD47', edgecolor='black', zorder=5)
ax.set(xlabel='Epoch', ylabel='Standardized MSE loss', title='Frozen-embedding head training')
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / 'training_validation_loss.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
@torch.inference_mode()
def predict_from_embeddings(embeddings):
    head.eval()
    values = torch.from_numpy(prepare_embeddings(embeddings)).to(DEVICE)
    scaled_prediction = head(values).cpu().numpy().ravel()
    return scaled_prediction * target_std + target_mean

test_prediction = predict_from_embeddings(test_embedding)
test_actual = test_data.BP.to_numpy()
test_results = test_data[['SMILES', 'BP']].copy()
test_results['predicted_BP_K'] = test_prediction
test_results['absolute_error_K'] = (test_results.BP - test_results.predicted_BP_K).abs()
test_results.to_csv(OUTPUT / 'test_predictions.csv', index=False)

test_loss = mean_loss(test_loader)
metrics = pd.DataFrame([{'selected_epoch': best_epoch, 'validation_loss': best_validation_loss, 'test_loss': test_loss,
                         'test_MAE_K': mean_absolute_error(test_actual, test_prediction),
                         'test_RMSE_K': mean_squared_error(test_actual, test_prediction) ** 0.5,
                         'test_R2': r2_score(test_actual, test_prediction)}])
display(metrics.round(3))

fig, ax = plt.subplots(figsize=(5.7, 5.2))
ax.scatter(test_actual, test_prediction, alpha=0.75, color='#4472C4', edgecolor='white', linewidth=0.35)
limits = [min(test_actual.min(), test_prediction.min()), max(test_actual.max(), test_prediction.max())]
ax.plot(limits, limits, '--', color='#C00000', label='Ideal prediction')
ax.set(xlim=limits, ylim=limits, xlabel='Measured boiling point (K)', ylabel='Predicted boiling point (K)',
       title='Held-out test-set parity plot')
ax.grid(alpha=0.25)
ax.legend()
ax.set_aspect('equal', adjustable='box')
fig.tight_layout()
fig.savefig(FIGURES / 'test_parity_plot.png', dpi=180, bbox_inches='tight')
plt.show()

good_examples = test_results.nsmallest(4, 'absolute_error_K')
bad_examples = test_results.nlargest(4, 'absolute_error_K')
display(good_examples.round(2))
display(bad_examples.round(2))

def display_molecule_examples(examples, title):
    molecules = [Chem.MolFromSmiles(smiles) for smiles in examples.SMILES]
    legends = [f'{smiles}\nactual {actual:.1f} K | predicted {predicted:.1f} K\nerror {error:.1f} K'
               for smiles, actual, predicted, error in zip(examples.SMILES, examples.BP,
                                                           examples.predicted_BP_K, examples.absolute_error_K)]
    print(title)
    display(Draw.MolsToGridImage(molecules, molsPerRow=4, subImgSize=(260, 200), legends=legends))

display_molecule_examples(good_examples, 'Lowest-error held-out examples')
display_molecule_examples(bad_examples, 'Highest-error held-out examples')

In [ ]:
def predict_boiling_point(smiles):
    """Predict a boiling point in K from one valid SMILES string using the selected head."""
    if not isinstance(smiles, str) or Chem.MolFromSmiles(smiles) is None:
        raise ValueError('Provide one valid SMILES string, for example CCO for ethanol.')
    embedding = embed_smiles([smiles], description='Embedding input SMILES')
    return float(predict_from_embeddings(embedding)[0])

SMILES_TO_PREDICT = 'CCO'  # Replace with a valid SMILES string.
predicted_bp = predict_boiling_point(SMILES_TO_PREDICT)
print(f'Predicted boiling point for {SMILES_TO_PREDICT}: {predicted_bp:.1f} K')